In [ ]:
using LinearAlgebra

In [ ]:
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

In [ ]:
H = [zeros(1,3); I];
T = [1  zeros(1,3);
     zeros(3,1) -I];

In [ ]:
function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I - hat(q[2:4])]
end

In [ ]:
function G(q)
    return L(q)*H
end

In [ ]:
function expq(ϕ)
    θ = norm(ϕ)
    
    #naive way with divide-by-zero badness
    #r = ϕ/θ
    #[cos(θ); r*sin(θ)]

    #using sinc, we can avoid divide-by-zero issues
    return [cos(θ); ϕ*sinc(θ/π)];
end

function logq(q)
    c = q[1]
    s = norm(q[2:4])
    θ = atan(s,c)
    return q[2:4]/sinc(θ/π)
end

In [ ]:
#sample a random rotation for initial attitude
ϕ = randn(3)
Q0 = exp(hat(ϕ))
q0 = expq(ϕ/2)

In [ ]:
#sanity check q -> Q
H'*L(q0)*R(q0)'*H

In [ ]:
H'*R(q0)'*L(q0)*H

In [ ]:
Q0

In [ ]:
#sample random angular velocity
ω0 = 0.1*randn(3)

In [ ]:
#Constant velocity
h = 0.1 #time step
n = 1 #number of time steps
tf = n*h #final time

In [ ]:
norm(ω0)

In [ ]:
#Assume constant velocity (exact solution)
q = L(q0)*expq(0.5*tf*ω0)
#q = L(q0)*[1; 0.5*tf*ω0] #Linear approximation
#q = q/norm(q)

In [ ]:
norm(q)

In [ ]:
#dynamics
function dynamics(x)
    q = x[1:4]
    ω = x[5:7]
    
    q̇ = 0.5*G(q)*ω
    #ω̇ = zeros(3)
    ω̇ = randn(3) #random walk

    ẋ = [q̇; ω̇]
end

In [ ]:
#Classic RK4 integrator: https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods
function rkstep(x)
    f1 = dynamics(x)
    f2 = dynamics(x + 0.5*h*f1)
    f3 = dynamics(x + 0.5*h*f2)
    f4 = dynamics(x + h*f3)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    #xn[1:4] .= xn[1:4]/norm(xn[1:4]) #re-normalize quaternion
    return xn
end

In [ ]:
#Simulate n time steps
x0 = [q0; ω0];
xk = x0
for k = 1:n
    xk = rkstep(xk)
end

In [ ]:
qn = xk[1:4]

In [ ]:
norm(qn)